# 01 — Exploration du dataset (raw)
## Objectif
Comprendre les colonnes, repérer les problèmes de qualité (valeurs manquantes, doublons, formats incohérents) et préparer un plan de nettoyage.

In [15]:
import pandas as pd
import numpy as np

In [16]:
file_path = "../DATA/RAW/dirty_cafe_sales.csv"

raw_data = pd.read_csv(file_path)

print("Nombre de lignes et colonnes :", raw_data.shape)
raw_data.head(10)

Nombre de lignes et colonnes : (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


In [17]:
print("Colonnes :", list(raw_data.columns))
raw_data.info()

Colonnes : ['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [18]:
raw_data.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [19]:
missing_count = raw_data.isna().sum()
missing_percent = (raw_data.isna().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
}).sort_values("missing_count", ascending=False)

missing_table[missing_table["missing_count"] > 0]

,missing_count,missing_percent
Location,3265,32.65
Payment Method,2579,25.79
Item,333,3.33
Price Per Unit,179,1.79
Total Spent,173,1.73
Transaction Date,159,1.59
Quantity,138,1.38


In [20]:
duplicate_rows = raw_data.duplicated().sum()
print("Nombre de lignes doublons :", duplicate_rows)

if duplicate_rows > 0:
    raw_data[raw_data.duplicated(keep=False)].head(20)

Nombre de lignes doublons : 0


In [21]:
text_columns = raw_data.select_dtypes(include="object").columns
print("Colonnes texte :", list(text_columns))

Colonnes texte : ['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


In [22]:
for col in text_columns:
    print("\n---", col, "---")
    print(raw_data[col].astype(str).str.strip().value_counts().head(10))


--- Transaction ID ---
Transaction ID
TXN_1961373    1
TXN_4831525    1
TXN_1228927    1
TXN_6486912    1
TXN_3447069    1
TXN_8219298    1
TXN_1010950    1
TXN_6376329    1
TXN_1897783    1
TXN_2767034    1
Name: count, dtype: int64

--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
nan          333
Name: count, dtype: int64

--- Quantity ---
Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
nan         138
Name: count, dtype: int64

--- Price Per Unit ---
Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
nan         179
UNKNOWN     164
Name: count, dtype: int64

--- Total Spent ---
Total Spent
6.0     979
12.0    939
3.0     930
4.0     923
20.0    746
15.0    734
8.0     677
10.0    524
2.0     497
9.0     479
Name: c

# Bilan : 
Dataset : 10 000 lignes, 8 colonnes

0 doublon 

Problème majeur : tout est lu en texte (object), même Quantity, Price Per Unit, Total Spent

Valeurs manquantes importantes :

Location : 32.65%

Payment Method : 25.79%

Valeurs “sales” repérées :

Total Spent contient "ERROR"

Transaction Date a des "UNKNOWN"

Payment Method et Location ont des "UNKNOWN" + des NaN